In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd
from corpus.speaker_room_dataset import SpeakerRoomDataset
import src.audio_transforms as at
import scipy.stats as stats

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer
from tqdm.auto import tqdm
import src.spatial_attn_architecture as attn_arch
import re

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!hostname

node084


In [3]:
### Get most recent config
config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=3-step=49432.ckpt"
old_style = False
### Get most recent config
# config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
# ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"
# old_style = True 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['hparas']['batch_size'] = 2 # config['data']['loader']['batch_size'] // args.gpus
config['num_workers'] = 2
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
# cue_duration = 0.5
# n_cue_frames = int(cue_duration * signal_sr)
# config['model']['n_cue_frames'] = n_cue_frames

config['cue_duration_test'] = True


In [4]:
# int(torch.__version__.split('.')[0])

In [5]:
# model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config,)
# model = model.eval()

In [6]:
## get layer norm parameters from model 
# model.model._orig_mod.model_dict.norm_coch_rep.weight.shape
# isinstance(model.model._orig_mod.model_dict.norm_coch_rep, torch.nn.LayerNorm)

# new_weight = model.model._orig_mod.model_dict.norm_coch_rep.state_dict()
# new_weight

In [7]:
from src.spatial_attn_architecture import SimpleAttentionalGain, BinauralAuditoryAttentionCNN, CNN2DExtractor
from src.layers import conv2d_same
from src.layers import padding as pad_utils
from src.custom_modules import HannPooling2d
import torch.nn as nn


class CueDurationCNNNew(BinauralAuditoryAttentionCNN):
    def __init__(self, input_sr, out_channels, kernel, stride, padding, pool_stride, pool_size, pool_padding, attn, dropout,
                  fc_size=512, global_avg_cue=False, num_classes={"num_words":800, "num_locs":504}, frequency_dim=40,
                  residual_attn=False, n_cue_frames=None, starting_output_len = 20000, norm_first=True, ln_affine=True, **kwargs):
        # 1. call parent constructor
        super(CueDurationCNNNew, self).__init__(input_sr, out_channels, kernel, stride, padding,
                                             pool_stride, pool_size, pool_padding, attn, dropout,
                                             fc_size, global_avg_cue, num_classes, frequency_dim,
                                             residual_attn, n_cue_frames, starting_output_len, norm_first,
                                             ln_affine, **kwargs)
        self.model_dict = nn.ModuleDict()
        self.layer_norm_params = {}
        self.output_height = frequency_dim
        self.output_len = starting_output_len # softcode eventually
        self.n_cue_frames = n_cue_frames
        # build architecture without nn.normalization functions 

        # add init norm params 
        self.layer_norm_params[f'norm_coch_rep'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'n_cue_frames': n_cue_frames}

        for idx in range(self.n_layers):
            nIn = self.input_channels if idx == 0 else out_channels[idx - 1]
            nOut = out_channels[idx]

            # Attentional block:
            if self.attn[idx] == 1:
                self.model_dict[f'attn{idx}'] = SimpleAttentionalGain(self.output_height, nIn, global_avg_cue=global_avg_cue, n_cue_frames=self.n_cue_frames)

            # pre-compute conv output sizes - will assign to self.output_height and self.output_len after defining block
            # Sizes will be used for normalization layers, but depend on order of norm and conv 
            # norm -> conv gets prior output shapes, conv -> norm gets new output shapes)
            # compute output shapes using conv formula [(Height + 2Pad - dilation * (kernel - 1) -1) /  Stride] + 1
            # ignoring dilation since it's not used in this model (dilation = 1)
            if self.padding[idx] == 'same':
                output_height = self.output_height
                output_len = self.output_len
            else:
                conv_padding, _ = pad_utils.get_padding_value(self.padding[idx], self.kernel[idx], stride=self.stride[idx])
                output_height = int(np.floor((self.output_height + (2 * conv_padding[0]) - (kernel[idx][0] - 1) - 1) / stride[idx][0]) + 1)
                output_len = int(np.floor((self.output_len + (2 * conv_padding[1]) -  (kernel[idx][1] - 1) - 1) / stride[idx][1]) + 1)
                if self.n_cue_frames:
                    n_cue_frames = int(np.floor((self.n_cue_frames + (2 * conv_padding[1]) -  (kernel[idx][1] - 1) - 1) / stride[idx][1]) + 1)

            block = nn.Sequential(conv2d_same.create_conv2d_pad(nIn, nOut, self.kernel[idx], stride=self.stride[idx], padding=self.padding[idx]),
                                  nn.ReLU())
            
            # init layer norm params as None - will populate if in checkpoint 
            self.layer_norm_params[f'layer_norm_{idx}'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'n_cue_frames': self.n_cue_frames if self.norm_first else n_cue_frames}
            # update post-conv init
            self.output_height, self.output_len, self.n_cue_frames = output_height, output_len, n_cue_frames
            self.model_dict[f'conv_block_{idx}'] = block

            if self.pool_stride[idx] != -1:
                self.model_dict[f'hann_pool_{idx}'] = HannPooling2d(stride=self.pool_stride[idx], pool_size=self.pool_size[idx], padding=self.pool_padding[idx])
                # Compute output shapes for pooling layers using conv formula [(Height - Filter + 2Pad)/ Stride]+1
                self.output_height = int(np.floor((self.output_height - pool_size[idx][0] + 2 * pool_padding[idx][0]) / pool_stride[idx][0]) + 1)
                self.output_len = int(np.floor((self.output_len - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)
                if self.n_cue_frames:
                    self.n_cue_frames = int(np.floor((self.n_cue_frames - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)


        self.output_size = self.output_height * nOut * self.output_len
        self.fullyconnected = nn.Linear(self.output_size, fc_size)
        self.relufc = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        # init ln params

        if self.dual_task:
            self.classificationWord = nn.Linear(fc_size, self.num_words)
            self.classificationLoc = nn.Linear(fc_size, self.num_locs)
        else:
            self.classification = nn.Linear(fc_size, self.num_classes)
    
    def forward(self, cue=None, mixture=None, cue_mask_ixs=None):
        cue = nn.functional.layer_norm(cue, cue.shape[1:],
                                       weight=self.layer_norm_params['norm_coch_rep']['cue_weight'], # [..., :  self.layer_norm_params[f'norm_coch_rep']['n_cue_frames']]
                                       bias=self.layer_norm_params['norm_coch_rep']['cue_bias'])
        
        attn = nn.functional.layer_norm(mixture, mixture.shape[1:], 
                                        weight=self.layer_norm_params['norm_coch_rep']['weight'], 
                                        bias=self.layer_norm_params['norm_coch_rep']['bias'])

        for idx in range(self.n_layers):
            if self.attn[idx] == 1:
                if self.residual_attn:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs) + attn
                else:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs)
                    
            # print(f"conv_block_{idx} post gain max and min: {attn.max().item(), attn.min().item()}")
            if self.norm_first:
                cue = nn.functional.layer_norm(cue, cue.shape[1:], 
                                               weight=self.layer_norm_params[f'layer_norm_{idx}']['cue_weight'],
                                               bias=self.layer_norm_params[f'layer_norm_{idx}']['cue_bias'])
                cue = self.model_dict[f'conv_block_{idx}'](cue)
                attn = nn.functional.layer_norm(attn, attn.shape[1:], 
                                                weight=self.layer_norm_params[f'layer_norm_{idx}']['weight'],
                                                bias=self.layer_norm_params[f'layer_norm_{idx}']['bias'])
                attn = self.model_dict[f'conv_block_{idx}'](attn)
            else:
                cue = self.model_dict[f'conv_block_{idx}'](cue)
                cue = nn.functional.layer_norm(cue, cue.shape[1:], 
                                               weight=self.layer_norm_params[f'layer_norm_{idx}']['cue_weight'],
                                               bias=self.layer_norm_params[f'layer_norm_{idx}']['cue_bias'])
                attn = self.model_dict[f'conv_block_{idx}'](attn)
                attn = nn.functional.layer_norm(attn, attn.shape[1:], 
                                                weight=self.layer_norm_params[f'layer_norm_{idx}']['weight'],
                                                bias=self.layer_norm_params[f'layer_norm_{idx}']['bias'])

            # print(f"conv_block_{idx} post conv and norm max and min: {attn.max().item(), attn.min().item()}")
            if self.pool_stride[idx] != -1:
                cue = self.model_dict[f'hann_pool_{idx}'](cue)
                attn = self.model_dict[f'hann_pool_{idx}'](attn)

        out = attn

        out = out.view(out.size(0), self.output_size) # B x FC size
        out = self.fullyconnected(out)        
        out = self.relufc(out)
        out = self.dropout(out)        
        if self.dual_task:
            word_out = self.classificationWord(out)
            loc_out = self.classificationLoc(out)
            return word_out, loc_out
        else:
            return self.classification(out)



In [8]:
class CueDurationCNN2DExtractor(CNN2DExtractor):
    def __init__(self, input_sr, out_channels, kernel, stride, padding, pool_stride, pool_size, pool_padding, attn, dropout,
                  fc_size=512, global_avg_cue=False, num_classes={"num_words":998, "num_locs":504}, residual_attn=False,
                  double_size=False, n_cue_frames=None, **kwargs):
        # 1. call parent constructor
        super(CueDurationCNN2DExtractor, self).__init__(input_sr, out_channels, kernel, stride, padding,
                                             pool_stride, pool_size, pool_padding, attn, dropout,
                                             fc_size, global_avg_cue, num_classes, residual_attn, double_size, n_cue_frames, **kwargs)
        self.model_dict = nn.ModuleDict()
        self.layer_norm_params = {}
        self.frequency_dim = 40

        self.output_height = self.frequency_dim
        self.output_len = 20000 # softcode eventually
        self.n_cue_frames = n_cue_frames
        # build architecture without nn.normalization functions 

        # add init norm params 
        self.layer_norm_params[f'norm_coch_rep'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'n_cue_frames': n_cue_frames}
        self.model_dict["attn_block_in"] = SimpleAttentionalGain(self.frequency_dim, self.input_channels, global_avg_cue=global_avg_cue, n_cue_frames=self.n_cue_frames)

        for idx in range(self.n_layers):
            nIn = self.input_channels if idx == 0 else out_channels[idx - 1]
            nOut = out_channels[idx]

            self.layer_norm_params[f'layer_norm_{idx}'] = {"weight": None, "bias": None, "cue_weight": None, "cue_bias": None, 'n_cue_frames': self.n_cue_frames}

            if self.pool_stride[idx] != -1:
                # print(f'nIn: {nIn}, nOut: {nOut}, kernel: {self.kernel[idx]}, stride: {self.stride[idx]}, padding: {self.padding[idx]}')

                block = nn.Sequential(
                                    conv2d_same.create_conv2d_pad(nIn, nOut, self.kernel[idx], stride=self.stride[idx], padding=self.padding[idx]),
                                    nn.ReLU(),
                                    HannPooling2d(stride=self.pool_stride[idx], pool_size=self.pool_size[idx], padding=self.pool_padding[idx]))
            else:
                block = nn.Sequential(
                                    conv2d_same.create_conv2d_pad(nIn, nOut, self.kernel[idx], stride=self.stride[idx], padding=self.padding[idx]),
                                    nn.ReLU())

            self.model_dict[f'conv_block_{idx}'] = block

            # Compute output shapes using conv formula [(Height - Filter + 2Pad)/ Stride]+1
            if self.padding[idx] == 'same':
                pass
            else:
                self.output_height = int(np.floor((self.output_height - kernel[idx][0] + 2 * padding[idx][0]) / stride[idx][0]) + 1)
                self.output_len = int(np.floor((self.output_len -  kernel[idx][1] + 2 * padding[idx][1]) / stride[idx][1]) + 1)
                if self.n_cue_frames:
                    self.n_cue_frames = int(np.floor((self.n_cue_frames - kernel[idx][1] + 2 * padding[idx][1]) / stride[idx][1]) + 1)
            if self.pool_stride[idx] != -1:
                self.output_height = int(np.floor((self.output_height - pool_size[idx][0] + 2 * pool_padding[idx][0]) / pool_stride[idx][0]) + 1)
                self.output_len = int(np.floor((self.output_len - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)
                if self.n_cue_frames:
                    self.n_cue_frames = int(np.floor((self.n_cue_frames - pool_size[idx][1] + 2 * pool_padding[idx][1]) / pool_stride[idx][1]) + 1)
            # Attentional block:
            if self.attn[idx] == 1:
                self.model_dict[f'attn{idx}'] = SimpleAttentionalGain(self.output_height, nOut, global_avg_cue=global_avg_cue, n_cue_frames=self.n_cue_frames)
                
        self.output_size = self.output_height * nOut * self.output_len
        self.fullyconnected = nn.Linear(self.output_size, fc_size)
        self.relufc = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        if self.dual_task:
            self.classificationWord = nn.Linear(fc_size, self.num_words)
            self.classificationLoc = nn.Linear(fc_size, self.num_locs)
        else:
            self.classification = nn.Linear(fc_size, self.num_classes)
    
    def forward(self, cue=None, mixture=None, cue_mask_ixs=None):
        cue = nn.functional.layer_norm(cue, cue.shape[1:],
                                       weight=self.layer_norm_params['norm_coch_rep']['cue_weight'],
                                       bias=self.layer_norm_params['norm_coch_rep']['cue_bias'])
        
        attn = nn.functional.layer_norm(mixture, mixture.shape[1:], 
                                        weight=self.layer_norm_params['norm_coch_rep']['weight'], 
                                        bias=self.layer_norm_params['norm_coch_rep']['bias'])
        if self.residual_attn:
            attn = self.model_dict["attn_block_in"](cue, mixture, cue_mask_ixs) + mixture
        else:
            attn = self.model_dict["attn_block_in"](cue, mixture, cue_mask_ixs)

        for idx in range(self.n_layers):
            cue = nn.functional.layer_norm(cue, cue.shape[1:], 
                                            weight=self.layer_norm_params[f'layer_norm_{idx}']['cue_weight'],
                                            bias=self.layer_norm_params[f'layer_norm_{idx}']['cue_bias'])
            cue = self.model_dict[f'conv_block_{idx}'](cue)
            attn = nn.functional.layer_norm(attn, attn.shape[1:], 
                                            weight=self.layer_norm_params[f'layer_norm_{idx}']['weight'],
                                            bias=self.layer_norm_params[f'layer_norm_{idx}']['bias'])
            attn = self.model_dict[f'conv_block_{idx}'](attn)
            # print('mixture acts ',  attn)
            # print(f"conv_block_{idx} pre attn, {attn.max(), attn.min()}")
            if self.attn[idx] == 1:
                if self.residual_attn:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs) + attn
                else:
                    attn = self.model_dict[f'attn{idx}'](cue, attn, cue_mask_ixs)
                # print(f"conv_block_{idx} post attn, {attn.max(), attn.min()}")
        out = attn

        out = out.view(out.size(0), self.output_size) # B x FC size
        out = self.fullyconnected(out)        
        out = self.relufc(out)
        out = self.dropout(out)        
        if self.dual_task:
            word_out = self.classificationWord(out)
            loc_out = self.classificationLoc(out)
            return word_out, loc_out
        else:
            return self.classification(out)

In [9]:
config['model']

{'input_sr': 10000,
 'out_channels': [32, 64, 256, 512, 512, 512, 512],
 'kernel': [[2, 34], [2, 14], [5, 5], [5, 5], [6, 6], [5, 5], [6, 6]],
 'stride': [[1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1], [1, 1]],
 'padding': ['valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time',
  'valid_time'],
 'pool_stride': [[2, 4], [2, 4], [1, 5], [1, 4], [1, 1], [1, 1], [2, 4]],
 'pool_size': [[9, 13], [9, 13], [1, 13], [1, 13], [1, 1], [1, 1], [6, 13]],
 'pool_padding': [[4, 6], [4, 6], [0, 6], [0, 6], [0, 0], [0, 0], [3, 6]],
 'attn': [1, 1, 1, 1, 1, 1, 1],
 'num_classes': {'num_words': 800},
 'fc_size': 512,
 'global_avg_cue': False,
 'dropout': 0.6,
 'attn_constraints': {'slope': True},
 'norm_first': False,
 'ln_affine': False}

In [10]:

cue_dur_in_s = 2 
cue_dur = int(cue_dur_in_s * coch_sr) 


if old_style:
    model = CueDurationCNN2DExtractor(**config['model'], n_cue_frames=cue_dur)
else:
    model = CueDurationCNNNew(**config['model'], n_cue_frames=cue_dur)

# restore params from checkpoint 
checkpoint = torch.load(ckpt_path)
checkpoint['state_dict'] = {k.replace('model._orig_mod.', ''):v for k,v in checkpoint['state_dict'].items()}

# below is naming update to map checkpoint parameters to new model dict names
new_state_dict = {}
# old_style = True 
for k,v in  checkpoint['state_dict'].items():
    # print(k)
    if config['model'].get('ln_affine', True):
        # update key for easy norm layer access
        if 'conv' in k and '.0.' in k:
            k = k.replace('conv_block_', 'layer_norm_')
            k = k.replace('.0.', '.')
            print(k)
        # decrement conv layer ixs in dict to match model
        elif 'conv' in k and '.1.' in k:
            k = k.replace('.1.', '.0.')
        # decrement pool layer rixs in dict to match model
        elif 'conv' in k and '.3.' in k:
            k = k.replace('.3.', '.2.')
    # if old_style:
    #     if 'attn' in k:
    #         if 'in' in k:
    #             k = k.replace('_block_in', '0')
    #         # if digit in string, add 1 to that digit 
    #         else:
    #             print(k)
    #             layer_num = int(re.search('attn(-?\d+)', k).group(0).strip('attn'))
    #             k = k.replace(str(layer_num), str(layer_num + 1 ))

    new_state_dict[k] = v

norm_param_dict = {k:v for k,v in new_state_dict.items() if 'norm' in k}

num_classes={'num_words': 800}
Model performing word task
Conv block order: Conv -> ReLU -> LN
Using cue duration of 20000 frames
Using cue duration of 4992 frames
Using cue duration of 1245 frames
Using cue duration of 249 frames
Using cue duration of 62 frames
Using cue duration of 57 frames
Using cue duration of 53 frames
Using cue duration of 20000 frames
Using cue duration of 4992 frames
Using cue duration of 1245 frames
Using cue duration of 249 frames
Using cue duration of 62 frames
Using cue duration of 57 frames
Using cue duration of 53 frames


In [11]:
norm_param_dict.keys()

dict_keys(['model_dict.norm_coch_rep.weight', 'model_dict.norm_coch_rep.bias'])

In [12]:
for key, param in norm_param_dict.items():
    layer_name = key.split('.')[1]
    n_cue_frames_at_layer = model.layer_norm_params[f'{layer_name}']['n_cue_frames']
    param_type = 'weight' if 'weight' in key else 'bias'
    model.layer_norm_params[f'{layer_name}'][f'{param_type}'] = param
    model.layer_norm_params[f'{layer_name}'][f'cue_{param_type}'] = param[..., : n_cue_frames_at_layer]
        

In [13]:
#  model.layer_norm_params

In [13]:
model.load_state_dict(new_state_dict, strict=False) # strict=False to skip attn weights 



_IncompatibleKeys(missing_keys=[], unexpected_keys=['coch_gram.full_rep.rep.downsampling_op.kernel', 'coch_gram.full_rep.rep.Cochleagram.compute_rep.coch_filters', 'coch_gram.full_rep.rep.Cochleagram.downsampling.kernel', 'model_dict.norm_coch_rep.weight', 'model_dict.norm_coch_rep.bias'])

In [15]:
## Want to restore parameters from original model to new model
## For all layers, restore weights and biases from model_dict.conv_block_{idx}.norm to model_dict.mixture_norm_{idx} and model_dict.cue_norm_{idx}

# for k,v in model.model_dict.items():
#     print(k)
#     if 'norm' in k:



In [14]:
dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
                            excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
                            cue_type='voice_and_location',
                            sr=signal_sr) 


diotic_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), 
                    at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                    at.DuplicateChannel(),

            ])


diotic_transforms = diotic_transforms.cuda()


def single_signal_collate_fn(batch):
    #apply transforsms to batch
    cues = torch.stack([diotic_transforms(cue, None)[0] for cue, fg, bg, label, confusion in batch])
    mixtures = torch.stack([diotic_transforms(fg, bg)[0] for cue, fg, bg, label, confusion in batch]).type(torch.FloatTensor)
    labels = torch.tensor([label for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    confusion = torch.tensor([confusion for cue, fg, bg, label, confusion in batch]).type(torch.LongTensor)
    return cues, mixtures, labels, confusion


dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False, num_workers=config['num_workers'], collate_fn=single_signal_collate_fn)


In [15]:
# torch module to center crop the cochleagram to a given duration 
class CenterCropCoch(torch.nn.Module):
    def __init__(self, crop_duration, sig_duration, sr, pad_dur=False):
        super().__init__()
        self.n_crop_frames = int(crop_duration * sr)
        sig_frames = int(sig_duration * sr)
        self.crop_context = sig_frames - self.n_crop_frames
        self.crop_start = self.crop_context // 2
        self.crop_end = self.crop_start + self.n_crop_frames
        self.pad_to_dur = False
        
        if pad_dur > crop_duration:
            self.pad_to_dur = True 
            self.pad_context = int(pad_dur * sr) - self.n_crop_frames

    def forward(self, x):
        # crop x to n_crop_frames and zero pad back to original length
        cropped = x[..., self.crop_start:self.crop_end]
        if self.pad_to_dur:
            cropped = torch.nn.functional.pad(cropped, (0, self.pad_context), "constant", 0)
        return cropped 



In [18]:
!hostname

node084


In [19]:
%matplotlib inline
import matplotlib.pyplot as plt

# plt.imshow(cue[0][0].cpu().numpy(), aspect='auto', origin='upper')

In [20]:
# torch.load(ckpt_path)['state_dict'].keys()  

In [16]:
output_dict = {'results': None, 'confusions': None}
accuracies = []
pred_list = []
true_word_int = []
confusions_list  = []
all_probs_of_interest = []
guessed_both = []

crop_duration = cue_dur_in_s # in seconds 
sig_duration = 2 # in seconds 
center_crop = CenterCropCoch(crop_duration, sig_duration, coch_sr, pad_dur=0.5).cuda() 

signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
n_cue_frames = int(crop_duration * coch_sr)
print(n_cue_frames)
config['model']['n_cue_frames'] = n_cue_frames
#
# module = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False)
module = binaural_lightning.BinauralAttentionModule(config=config)
model = torch.compile(model)

model = model.eval().cuda()
coch_gram = module.coch_gram.cuda()

with torch.no_grad(): 
    for ix, batch in enumerate(tqdm(dataloader)):
        if ix > 10:
            break
        cue, mixture, label, confusion = batch
        # cue, fg, label = batch
        # just running diotically on fg 
        # cue, _ = audio_transforms(cue.cuda(), None)
        # mixture, _ = diotic_transforms(fg.cuda().squeeze(), None)
        # cue = cue.cuda()
        # mixture = mixture.cuda()
        cue, mixture = coch_gram(cue.cuda(), mixture.cuda())
        cue = center_crop(cue)
        # cue_mask_ixs = torch.arange(cue.shape[0])
        logits = model(cue, mixture, None)
        probs = logits.softmax(dim=-1).cpu().detach().numpy()

        # get top 2 probs for each example
        top_2_probs = torch.topk(logits.softmax(-1), 5, dim=-1).indices.cpu().detach().numpy()
        return_both = (np.isin(label.numpy(), top_2_probs) * np.isin(confusion.numpy(), top_2_probs)).astype('int')
        
        targ_probs = probs[torch.arange(probs.shape[0]), label]
        conf_probs = probs[torch.arange(probs.shape[0]), confusion]
        probs_of_interest = np.concatenate([targ_probs[:, None], conf_probs[:, None]], axis=1)

        preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
        true_word = label.numpy().astype('int')
        accuracy = (preds == true_word).astype('int')
        conf = (preds == confusion.numpy().astype('int')).astype('int')
        confusions_list.append(conf)
        accuracies.append(accuracy)
        pred_list.append(preds)
        true_word_int.append(true_word)
        all_probs_of_interest.append(probs_of_interest)
        guessed_both.append(return_both)
        
accuracies = np.concatenate(accuracies)
pred_list = np.concatenate(pred_list)
true_word_int = np.concatenate(true_word_int)
confusions_list = np.concatenate(confusions_list)
all_probs_of_interest = np.concatenate(all_probs_of_interest, axis=0)
guessed_both = np.concatenate(guessed_both)

output_dict['probs_of_interest'] = all_probs_of_interest
output_dict['results'] = accuracies
output_dict['preds'] = pred_list
output_dict['true_word_int'] = true_word_int

print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

20000
Using explicit dim specificaion for demeaning in audio transforms
Using cue durration test architecture
num_classes={'num_words': 800}
Model performing word task
Conv block order: Conv -> ReLU -> LN
Using cue duration of 20000 frames
Using cue duration of 4992 frames
Using cue duration of 1245 frames
Using cue duration of 249 frames
Using cue duration of 62 frames
Using cue duration of 57 frames
Using cue duration of 53 frames
Using cue duration of 20000 frames
Using cue duration of 4992 frames
Using cue duration of 1245 frames
Using cue duration of 249 frames
Using cue duration of 62 frames
Using cue duration of 57 frames
Using cue duration of 53 frames


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


  6%|▌         | 11/198 [00:27<07:41,  2.47s/it]


Accuracy using 2s cue: 0.59, (0.04)
Confusions using 2s cue: 0.09, (0.02)
Guessed both talker words: 0.10, (0.02)


Process ForkProcess-2:
Process ForkProcess-1:
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/concurrent/futures/process.py", line 244, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiprocessing/synchronize.py", line 95, in __enter__
    return self._semlock.__enter__()
           ^^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/multiproc

Using cue duration of 2 seconds    
Accuracy using fg as cue: 0.56, (0.01)    
Confusions using fg as cue: 0.09, (0.00)    
Guessed both talker words: 0.12, (0.01)    

Using cue duration of 0.1 seconds    
Accuracy using fg as cue: 0.50, (0.01)    
Confusions using fg as cue: 0.09, (0.01)    
Guessed both talker words: 0.11, (0.01)

In [21]:
# print(f"Accuracy using {crop_duration}s cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using {crop_duration}s cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")

In [22]:
# print(f"Using cue duration of {cue_duration} seconds")
# print(f"Accuracy using fg as cue: {np.mean(accuracies):.2f}, ({stats.sem(accuracies):.2f})")
# print(f"Confusions using fg as cue: {np.mean(confusions_list):.2f}, ({stats.sem(confusions_list):.2f})")
# print(f"Guessed both talker words: {np.mean(guessed_both):.2f}, ({stats.sem(guessed_both):.2f})")